# 10. Futility-анализ (ранняя остановка по бесперспективности)


## 0. Постановка задачи

Есть две разновидности ранней остановки эксперимента:

- **efficacy** — эффект уже доказан: статистика пересекла верхнюю границу. Это материал лекции 09 (α-spending) — останавливаемся, потому что *победили*.
- **futility** — эффекта, скорее всего, нет или он меньше MDE. Останавливаемся, чтобы не жечь трафик на заведомо «мёртвый» тест.

Формальный вопрос, на который отвечает эта лекция:

> Если на промежуточном взгляде эффект выглядит слабее MDE, с какой вероятностью тест всё-таки дойдёт до значимости к концу?

Если эта вероятность мала — тратить оставшийся трафик бессмысленно. Именно её мы научимся считать (разделы 2–4), а затем переведём её в правило остановки (разделы 4–5).

**Ключевая асимметрия**, которую мы докажем в разделе 7: peeking по эффективности **раздувает** ошибку I рода, а futility-остановка ошибку I рода **не раздувает** — она лишь снижает мощность. Поэтому futility — «безопасная» остановка, и её можно добавлять к тесту почти без риска для достоверности результата.


In [ ]:
import math
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm
from scipy.optimize import brentq
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 12

SEED = 12345
rng = np.random.default_rng(SEED)

# Единая палитра исходов на всю лекцию
C_FUT = "#d95f02"   # оранжевый — futility (бесперспективность)
C_EFF = "#1b9e77"   # зелёный — efficacy (эффект есть)
C_CONT = "#999999"  # серый — продолжаем


In [ ]:
@dataclass(frozen=True)
class Design:
    """Параметры сквозного примера: A/B-тест непрерывной метрики (средний чек)."""
    alpha: float = 0.05        # односторонняя ошибка I рода
    beta: float = 0.20         # ошибка II рода (power = 0.80)
    sigma: float = 100.0       # СКО метрики, руб.
    mu0: float = 500.0         # среднее в контроле
    mde: float = 10.0          # абсолютный MDE, руб.
    K: int = 5
    t: tuple = (0.30, 0.50, 0.70, 0.85, 1.0)  # доли информации (моменты взглядов)

    @property
    def delta(self) -> float:
        """Стандартизованный MDE: delta = mde / sigma."""
        return self.mde / self.sigma

    @property
    def theta_mde(self) -> float:
        """Дрейф (нецентральность) под MDE: theta = z_{1-alpha} + z_{1-beta}."""
        return norm.ppf(1 - self.alpha) + norm.ppf(1 - self.beta)

    @property
    def i_max(self) -> float:
        """Максимальная информация: I_max = (theta_mde / delta)^2."""
        return (self.theta_mde / self.delta) ** 2

    @property
    def n_max(self) -> float:
        """Максимальный размер выборки на группу (двухвыборочный z-тест)."""
        return 2 * self.sigma ** 2 * self.i_max


design = Design()
design


### 0.1. Сквозной пример и «цена» продолжения теста

Через всю лекцию пройдёт один пример, чтобы все формулы и графики были про одно и то же. Метрика — **средний чек** (непрерывная, $\sigma = 100$ руб.), контроль $\mu_0 = 500$ руб. Хотим поймать подъём $MDE = 10$ руб. с мощностью 80% при одностороннем $\alpha = 5\%$.

Смысл таблицы ниже — почувствовать масштаб: полный тест «стоит» десятки миллионов показов, и каждый процент ранней остановки — это реально сэкономленный трафик, который можно отдать следующему эксперименту.


In [ ]:
n = int(np.ceil(design.n_max))
print(f"Стандартизованный MDE:  delta = {design.delta:.3f}")
print(f"Дрейф под MDE:          theta_mde = {design.theta_mde:.3f}")
print(f"Размер на группу (макс): n_max = {n:,} пользователей")
print(f"Суммарный трафик теста:  {2*n:,} пользователей")

cost = pd.DataFrame({
    "Момент остановки": ["25%", "50%", "75%", "100%"],
    "Доля трафика": [0.25, 0.50, 0.75, 1.00],
    "Пользователей": [int(2 * n * x) for x in (0.25, 0.50, 0.75, 1.00)],
})
cost["Сэкономлено vs 100%"] = (2 * n - cost["Пользователей"]).astype(int)
cost


## 1. Аппарат: стандартизованная статистика и информация

Вся futility-механика живёт на одной величине — стандартизованной статистике $Z_t$ — и на доле информации $t$. Главное достоинство этой постановки: **любая метрика и любой момент времени сводятся к одному безразмерному числу** с дисперсией ровно 1. Поэтому все формулы ниже не зависят от того, что мы меряем — чек, конверсию или время на странице.

Ниже — полные выводы. Их можно читать подряд, а можно разворачивать по мере необходимости.

<details>
<summary>1.1. Вывод распределения $Z_t$ (развернуть)</summary>

Данные: контроль $A$ и тест $B$, по $n_t$ наблюдений каждая, независимые, из нормальных распределений с общей дисперсией $\sigma^2$. Оценка эффекта — разность средних: $\hat\delta_t=\bar X_B-\bar X_A$.

**Дисперсия среднего.** Разберём по шагам, почему $\mathrm{Var}(\bar X_A)=\sigma^2/n_t$:

$$\mathrm{Var}(\bar X_A)=\mathrm{Var}\!\left(\frac1{n_t}\sum_{i=1}^{n_t}X_{A,i}\right)=\frac{1}{n_t^2}\sum_{i=1}^{n_t}\mathrm{Var}(X_{A,i})=\frac{1}{n_t^2}\cdot n_t\sigma^2=\frac{\sigma^2}{n_t}.$$

Здесь первое равенство — определение среднего, второе — $\mathrm{Var}(cY)=c^2\mathrm{Var}(Y)$ плюс независимость наблюдений (ковариации равны нулю), третье — все наблюдения имеют одну дисперсию $\sigma^2$. Аналогично $\mathrm{Var}(\bar X_B)=\sigma^2/n_t$.

**Дисперсия разности.** Так как выборки независимы, дисперсии складываются:

$$\mathrm{Var}(\hat\delta_t)=\mathrm{Var}(\bar X_B-\bar X_A)=\frac{\sigma^2}{n_t}+\frac{\sigma^2}{n_t}=\frac{2\sigma^2}{n_t}.$$

Стандартная ошибка $SE_t=\sigma\sqrt{2/n_t}$.

**Стандартизация.** Делим оценку на *свою* ошибку: $Z_t=\hat\delta_t/SE_t$. Тогда

$$\mathbb E[Z_t]=\frac{\mathbb E[\hat\delta_t]}{SE_t}=\frac{\delta}{SE_t},\qquad \mathrm{Var}(Z_t)=\frac{\mathrm{Var}(\hat\delta_t)}{SE_t^2}=\frac{2\sigma^2/n_t}{2\sigma^2/n_t}=1.$$

Итог: $Z_t\sim\mathcal N\big(\delta/SE_t,\,1\big)$. $Z_t$ безразмерен, дисперсия всегда 1 — это «линейка», по которой сравнимы любые тесты. Вся «разнокалиберность» метрик спрятана в $SE_t$.

**Вводим информацию.** Это обратная дисперсия оценки: $I_t=1/SE_t^2=n_t/(2\sigma^2)$. Тогда среднее переписывается:

$$\mathbb E[Z_t]=\frac{\delta}{SE_t}=\delta\sqrt{I_t}.$$

Введём два безразмерных числа — долю информации $t$ и дрейф $\theta$:

$$t=\frac{I_t}{I_{max}}\approx\frac{n_t}{n_{max}},\qquad \theta=\delta\sqrt{I_{max}}.$$

Тогда $\delta\sqrt{I_t}=\delta\sqrt{I_{max}}\cdot\sqrt{I_t/I_{max}}=\theta\sqrt t$, и компактно:

$$\boxed{Z_t\sim\mathcal N(\theta\sqrt t,\,1)}$$

</details>

<details>
<summary>1.2. Откуда $\theta_{MDE}=z_{1-\alpha}+z_{1-\beta}$ (развернуть)</summary>

Это не «константа из воздуха», а решение уравнения на мощность. При $t=1$ финальная статистика $Z_1\sim\mathcal N(\theta,1)$. Мощность — вероятность отвергнуть при истинном эффекте $\theta$:

$$\text{power}=P(Z_1>z_{1-\alpha})=P\!\left(\frac{Z_1-\theta}{1}>z_{1-\alpha}-\theta\right)=1-\Phi(z_{1-\alpha}-\theta).$$

Здесь $(Z_1-\theta)/1$ — стандартная нормальная. Требуем мощность $=1-\beta$:

$$1-\Phi(z_{1-\alpha}-\theta)=1-\beta\iff\Phi(z_{1-\alpha}-\theta)=\beta\iff z_{1-\alpha}-\theta=-z_{1-\beta}.$$

Последний переход: по определению квантиля $\Phi(-z_{1-\beta})=\beta$. Отсюда

$$\boxed{\theta_{MDE}=z_{1-\alpha}+z_{1-\beta}}$$

</details>

<details>
<summary>1.3. Независимые приращения и ковариация (развернуть)</summary>

Нужны два факта, из которых строится всё остальное.

**Факт 1. Оценка в момент $t$ раскладывается на старый и новый куски.** Распишем среднее по $n_t$ наблюдениям как сумму по первым $n_s$ и остальным $n_t-n_s$:

$$\bar X_B^{(t)}=\frac1{n_t}\sum_{i=1}^{n_t}X_{B,i}=\frac1{n_t}\left(\sum_{i=1}^{n_s}X_{B,i}+\sum_{i=n_s+1}^{n_t}X_{B,i}\right)=\frac{n_s}{n_t}\bar X_B^{(s)}+\frac{n_t-n_s}{n_t}\bar X_B^{(new)}.$$

То же для $A$, вычитаем: $\hat\delta_t=\frac{n_s}{n_t}\hat\delta_s+\frac{n_t-n_s}{n_t}\hat\delta_{new}$. Делим на $SE_t$.

**Факт 2. Каждый кусок пересчитывается в свою нормированную статистику.** Для старого куска $\hat\delta_s/SE_t=(\hat\delta_s/SE_s)\cdot(SE_s/SE_t)=Z_s\cdot(SE_s/SE_t)$, и

$$\frac{SE_s}{SE_t}=\frac{\sigma\sqrt{2/n_s}}{\sigma\sqrt{2/n_t}}=\sqrt{\frac{n_t}{n_s}}.$$

Обозначим $r=n_s/n_t$. Полный вес старого куска: $(n_s/n_t)\cdot\sqrt{n_t/n_s}=r\cdot r^{-1/2}=r^{1/2}=\sqrt{s/t}$. Для нового куска: доля $1-r$, поправка $(1-r)^{-1/2}$, итого вес $\sqrt{(t-s)/t}$. Получаем

$$\boxed{Z_t=\sqrt{\frac{s}{t}}\,Z_s+\sqrt{\frac{t-s}{t}}\,Z_{new}}$$

где $Z_{new}=\hat\delta_{new}/SE_{new}$ — свежая независимая выборка, поэтому $Z_{new}\perp Z_s$ и $\mathrm{Var}(Z_{new})=1$.

**Проверка нормировки** (сумма квадратов весов = 1): $s/t+(t-s)/t=1$, значит $\mathrm{Var}(Z_t)=1$. ✓

**Проверка средних**: $\mathbb E[Z_{new}]=\theta\sqrt{t-s}$, поэтому

$$\mathbb E[Z_t]=\sqrt{\frac{s}{t}}\,\theta\sqrt s+\sqrt{\frac{t-s}{t}}\,\theta\sqrt{t-s}=\theta\,\frac{s+(t-s)}{\sqrt t}=\theta\sqrt t.\;\checkmark$$

**Ковариация двух взглядов** (по билинейности):

$$\mathrm{Cov}(Z_s,Z_t)=\sqrt{\frac{s}{t}}\,\mathrm{Cov}(Z_s,Z_s)+\sqrt{\frac{t-s}{t}}\,\mathrm{Cov}(Z_s,Z_{new})=\sqrt{\frac{s}{t}}\cdot1+0=\sqrt{\frac{s}{t}}.$$

Так как обе дисперсии равны 1, ковариация совпадает с корреляцией: $\mathrm{corr}(Z_s,Z_t)=\sqrt{s/t}$, то есть $R^2=s/t$ — доля общей информации.

</details>

<details>
<summary>1.4. Переход к $B(t)$ и его физический смысл (развернуть)</summary>

**Зачем нужен переход.** В $Z$-единицах две неприятности: дисперсия всегда 1, а приращение $Z_t-Z_s$ имеет дисперсию

$$\mathrm{Var}(Z_t-Z_s)=1+1-2\,\mathrm{Cov}(Z_s,Z_t)=2-2\sqrt{s/t},$$

которая зависит от *обоих* моментов, а не от длины интервала. Устраним это, умножив на $\sqrt t$: $B(t)=\sqrt t\,Z_t$.

**Свойства (все выводятся из $Z_t\sim\mathcal N(\theta\sqrt t,1)$).**

$$\mathbb E[B(t)]=\theta t,\qquad \mathrm{Var}[B(t)]=t,$$
$$\mathrm{Cov}(B(s),B(t))=\sqrt{st}\,\mathrm{Cov}(Z_s,Z_t)=s=\min(s,t),$$
$$\mathrm{Cov}(B(s),B(t)-B(s))=\mathrm{Cov}(B(s),B(t))-\mathrm{Var}(B(s))=s-s=0.$$

Так как $(B(s),B(t))$ совместно нормальны (линейные комбинации независимых нормальных приращений), нулевая ковариация означает **независимость**:

$$B(t)-B(s)\perp B(s),\qquad B(t)-B(s)\sim\mathcal N(\theta(t-s),\,t-s).$$

На шкале $B$ разложение — простое сложение без весов: $B(t)=B(s)+\Delta B$.

**Что такое $B$ физически.** Подставим $Z_t=\hat\delta_t\sqrt{I_t}$ (так как $SE_t=1/\sqrt{I_t}$):

$$B(t)=\sqrt t\,Z_t=\sqrt{\frac{I_t}{I_{max}}}\cdot\hat\delta_t\sqrt{I_t}=\frac{\hat\delta_t I_t}{\sqrt{I_{max}}}.$$

Раскроем $\hat\delta_t I_t$ через данные ($\Delta_i=X_{B,i}-X_{A,i}$):

$$\hat\delta_t I_t=\left(\frac1{n_t}\sum_{i=1}^{n_t}\Delta_i\right)\frac{n_t}{2\sigma^2}=\frac{1}{2\sigma^2}\sum_{i=1}^{n_t}\Delta_i.$$

Итого

$$B(t)=\frac{1}{2\sigma^2\sqrt{I_{max}}}\sum_{i=1}^{n_t}\Delta_i.$$

$B$ — с точностью до константы **простая накопленная сумма по пользователям**. Поэтому независимость приращений — не приближение, а свойство дизайна: прирост — сумма по новым пользователям, независимая от суммы по старым. $B$ — броуновское движение с дрейфом $\theta$.

**Переход границ между шкалами.** Правило $Z_k>c_k$ домножением на $\sqrt{t_k}$ превращается в $B(t_k)>c_k\sqrt{t_k}=:b_k$. Вся техника работает на шкале $B$, а результаты возвращаются на $Z$ делением на $\sqrt{t_k}$.

</details>


In [ ]:
def z_stat(xA: np.ndarray, xB: np.ndarray, sigma: float) -> float:
    """Двухвыборочный z-тест непрерывной метрики (известная дисперсия)."""
    nA, nB = xA.size, xB.size
    se = sigma * np.sqrt(1 / nA + 1 / nB)
    return (xB.mean() - xA.mean()) / se


def simulate_b(rng, theta: float, t: np.ndarray, n_sim: int = 200) -> np.ndarray:
    """Траектории B(t) = sqrt(t)*Z_t — броуновское движение с дрейфом theta.

    Каждый шаг генерирует независимое нормальное приращение
    B(t_k) - B(t_{k-1}) ~ N(theta * dt, dt),  dt = t_k - t_{k-1}.
    """
    dt = np.diff(np.concatenate(([0.0], t)))
    inc = rng.normal(loc=theta * dt, scale=np.sqrt(dt), size=(n_sim, len(t)))
    return np.cumsum(inc, axis=1)


In [ ]:
# Проверка формулы Z_t ~ N(theta*sqrt(t), 1) на симуляции:
# убеждаемся, что среднее и дисперсия статистики совпадают с теорией.
t = np.asarray(design.t)
Z = simulate_b(rng, design.theta_mde, t, 20000) / np.sqrt(t)
print("Эмпирическое среднее Z_t под H1:", np.round(Z.mean(axis=0), 3))
print("Теория  theta*sqrt(t):           ", np.round(design.theta_mde * np.sqrt(t), 3))
print("Эмпирическое SD (должно быть ~1):", np.round(Z.std(axis=0), 3))


In [ ]:
# График нужен, чтобы "увидеть" дрейф: под H0 траектории топчутся у нуля,
# под H1 — систематически уходят вверх со скоростью theta*t.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4), sharey=True)

cases = [(0.0, "H0: эффекта нет (θ = 0)", "траектории топчутся около нуля"),
         (design.theta_mde, "H1: эффект = MDE", "траектории уходят вверх с дрейфом θt")]

for ax, (theta, title, note) in zip(axes, cases):
    B = simulate_b(rng, theta, t, 150)
    ax.plot(t, B.T, color=C_CONT, alpha=0.25, lw=0.9)
    ax.plot(t, theta * t, color="black", lw=2.2, ls="--", label=f"средний дрейф θt = {theta:.2f}t")
    ax.set_title(f"{title}\n{note}", fontsize=12)
    ax.set_xlabel("доля информации t")
    ax.legend(loc="upper left")

axes[0].set_ylabel("B(t) = √t · Z_t")
fig.suptitle("Броуновское движение с дрейфом — это и есть наш процесс наблюдений", fontsize=13.5)
fig.tight_layout()
plt.show()


## 2. Conditional Power (CP): условная вероятность «добить» тест

Теперь ответим на главный вопрос futility: *«если сейчас эффект выглядит слабо, с какой вероятностью тест всё-таки дойдёт до значимости к концу?»*. Ответ — **conditional power** (CP): вероятность отвергнуть $H_0$ в конце при условии текущих данных и при **заданном** предположении об истинном эффекте $\theta$.

**Почему это правильная величина для решения о futility.** Решение «продолжать или остановиться» — это ставка на будущее. CP ровно это и измеряет: шанс, что будущие данные доведут тест до значимости. Если шанс мал — тратить трафик бессмысленно.

<details>
<summary>2.1. Полный вывод формулы (развернуть)</summary>

**Шаг 1.** Выразим финальную статистику через текущую и приращение (на шкале $B$, по разделу 1.4):

$$Z_1=B(1)=B(t)+\big(B(1)-B(t)\big)=\sqrt t\,z_t+\big(B(1)-B(t)\big).$$

**Шаг 2.** Приращение независимо от $B(t)$ и распределено как $\mathcal N(\theta(1-t),\,1-t)$. Поэтому при условии $Z_t=z_t$ первое слагаемое — детерминированное число $\sqrt t\,z_t$, а второе — случайное. Условное распределение:

$$Z_1\mid Z_t=z_t\;\sim\;\mathcal N\big(\sqrt t\,z_t+\theta(1-t),\;1-t\big).$$

**Шаг 3.** Вероятность попасть в критическую область $\{Z_1>z_{1-\alpha}\}$ — хвост этого распределения. Стандартизуем: $\mu=\sqrt t z_t+\theta(1-t)$, $\sigma=\sqrt{1-t}$:

$$\mathrm{CP}(\theta)=P\!\left(\frac{Z_1-\mu}{\sigma}>\frac{z_{1-\alpha}-\mu}{\sigma}\right)=1-\Phi\!\left(\frac{z_{1-\alpha}-\sqrt t\,z_t-\theta(1-t)}{\sqrt{1-t}}\right).$$

$$\boxed{\mathrm{CP}(\theta)=1-\Phi\!\left(\frac{z_{1-\alpha}-\sqrt t\,z_t-\theta(1-t)}{\sqrt{1-t}}\right)}$$

**Структура формулы.** Числитель — «насколько финальная граница далеко от ожидаемого финала», знаменатель — «сколько неопределённости ещё осталось».

**Монотонность по $z_t$ (понадобится в разделе 3).** Обозначим $u=(z_{1-\alpha}-\sqrt t\,z_t-\theta(1-t))/\sqrt{1-t}$. Тогда

$$\frac{d\,\mathrm{CP}}{dz_t}=-\Phi'(u)\cdot\frac{du}{dz_t}=-\varphi(u)\cdot\left(-\frac{\sqrt t}{\sqrt{1-t}}\right)=\varphi(u)\sqrt{\frac{t}{1-t}}>0.$$

Так как плотность $\varphi(u)>0$, CP строго возрастает по $z_t$ — значит, уравнение $\mathrm{CP}=\gamma$ имеет единственный корень.

**Предельные случаи.**

- $t\to0$: $\sqrt t z_t\to0$, $\theta(1-t)\to\theta$, $\sqrt{1-t}\to1$, поэтому $\mathrm{CP}\to1-\Phi(z_{1-\alpha}-\theta)$ — априорная мощность дизайна.
- $t\to1$: знаменатель $\sqrt{1-t}\to0$. При $z_t<z_{1-\alpha}$ числитель $\to z_{1-\alpha}-z_t>0$, то есть $u\to+\infty$, $\mathrm{CP}\to0$; при $z_t>z_{1-\alpha}$ числитель отрицателен, $\mathrm{CP}\to1$. Получается ступенька в точке $z_t=z_{1-\alpha}$ — судьба уже решена.

</details>

**Выбор $\theta$ — самый тонкий момент.** Три осмысленных варианта:

- $\theta = 0$ (под $H_0$): «даже если эффекта нет, дойдём ли до ложной значимости?» — консервативно, редко останавливает.
- $\theta = \hat\theta_t = z_t/\sqrt t$ (текущий тренд): «считаем, что будущее похоже на уже увиденное». Заманчиво, но **оптимистично**: ранняя оценка шумная, и случайно «выстреливший» эффект кажется многообещающим.
- $\theta = \theta_{MDE}$ (дизайновый эффект): «если эффект ровно такой, как мы закладывали в power-расчёте». Стандартный ориентир.

Правило futility: **если $\mathrm{CP} < \gamma$ — останавливаемся** (типично $\gamma \in [0.10, 0.20]$).


In [ ]:
def conditional_power(z_t: float, t: float, theta: float, z_alpha: float = 1.6448536) -> float:
    """CP: вероятность Z_1 > z_alpha при наблюдаемом z_t и истинном дрейфе theta.

    Формула: 1 - Phi( (z_alpha - sqrt(t)*z_t - theta*(1-t)) / sqrt(1-t) ).
    """
    return 1.0 - norm.cdf((z_alpha - np.sqrt(t) * z_t - theta * (1 - t)) / np.sqrt(1 - t))


In [ ]:
z_alpha = norm.ppf(1 - design.alpha)
z_grid = np.linspace(-3, 3, 300)
tt = 0.5   # середина теста

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(z_grid, conditional_power(z_grid, tt, 0.0, z_alpha),
        color=C_CONT, lw=2.2, label="CP под H0 (θ = 0) — консервативно")
ax.plot(z_grid, conditional_power(z_grid, tt, design.theta_mde, z_alpha),
        color=C_EFF, lw=2.2, label="CP под MDE (θ = θ_MDE) — дизайновый ориентир")
ax.plot(z_grid, conditional_power(z_grid, tt, z_grid / np.sqrt(tt), z_alpha),
        color=C_FUT, lw=2.2, label="CP под трендом (θ = z_t/√t) — оптимистично")

ax.axhline(0.15, ls="--", color="black", lw=1.2)
ax.text(2.6, 0.16, "порог γ = 0.15", ha="right", fontsize=11)
ax.fill_between(z_grid, 0, 0.15, color=C_FUT, alpha=0.08)

z_cut = brentq(lambda z: conditional_power(z, tt, design.theta_mde, z_alpha) - 0.15, -3, 3)
ax.axvline(z_cut, color=C_FUT, ls=":", lw=1.3)
ax.annotate(f"при z_t < {z_cut:.2f} → стоп (CP<0.15)",
            xy=(z_cut, 0.05), xytext=(z_cut - 2.0, 0.30),
            arrowprops=dict(arrowstyle="->", color=C_FUT), color=C_FUT, fontsize=11)

ax.set_xlabel("наблюдаемая статистика $z_t$")
ax.set_ylabel("conditional power")
ax.set_title(f"CP при t = {tt}: выбор θ кардинально меняет ответ")
ax.legend(loc="upper left")
plt.show()


### 2.2. Карта CP: как шансы меняются с течением теста

График выше — срез карты CP в одной точке $t$. Вся карта на плоскости $(t, z)$ наглядно показывает, **почему время взгляда так важно**.

На ранних взглядах карта почти плоская — впереди ещё много данных, и почти любое текущее $z$ даёт средние шансы (судьбу решает будущее, а не прошлое). К концу карта «заостряется» в ступеньку — судьба почти решена. Контур $\mathrm{CP} = \gamma$ — это и есть граница «стоп / продолжаем».


In [ ]:
T, Zz = np.meshgrid(np.linspace(0.05, 0.95, 120), np.linspace(-3, 3, 240))
CP = conditional_power(Zz, T, design.theta_mde, z_alpha)

fig, ax = plt.subplots(figsize=(9, 5.5))
cs = ax.contourf(T, Zz, CP, levels=30, cmap="viridis")
ct = ax.contour(T, Zz, CP, levels=[0.15], colors="red", linewidths=2.5)
ax.clabel(ct, fmt={0.15: "CP = 0.15"}, fontsize=11)

ax.text(0.75, -2.2, "СТОП (futility)\nCP < 0.15", color="white", fontsize=12, ha="center", weight="bold")
ax.text(0.75, 2.2, "продолжаем", color="black", fontsize=12, ha="center", weight="bold")

ax.set_xlabel("доля информации t")
ax.set_ylabel("наблюдаемая статистика $z_t$")
ax.set_title("Conditional power под MDE: красный контур — граница futility-остановки")
cbar = fig.colorbar(cs, ax=ax)
cbar.set_label("conditional power")
plt.show()


In [ ]:
def plot_cp_interactive():
    def _plot(t, z_t, theta, alpha):
        za = norm.ppf(1 - alpha)
        cp = conditional_power(z_t, t, theta, za)
        fig, ax = plt.subplots(figsize=(8, 2.6))
        ax.barh(0, cp, color=C_FUT if cp < 0.15 else C_EFF)
        ax.set_xlim(0, 1); ax.set_yticks([])
        ax.axvline(0.15, ls="--", color="black")
        ax.set_title(f"CP = {cp:.3f}  →  {'СТОП (futility)' if cp < 0.15 else 'продолжаем'}")
        plt.show()
    interact(_plot,
             t=FloatSlider(value=0.3, min=0.05, max=0.95, step=0.05, description="t"),
             z_t=FloatSlider(value=0.5, min=-3, max=3, step=0.05, description="z_t"),
             theta=FloatSlider(value=design.theta_mde, min=0, max=4, step=0.1, description="θ"),
             alpha=FloatSlider(value=0.05, min=0.01, max=0.20, step=0.01, description="α"))
plot_cp_interactive()


## 3. Перевод порога CP в границу (инверсия)

Порог $\mathrm{CP}<\gamma$ можно переписать в терминах статистики. При фиксированных $t,\alpha,\gamma,\theta$ определим $a_{CP}$ как решение

$$\mathrm{CP}(a_{CP};t,\theta)=\gamma.$$

Так как CP монотонно растёт по $z_t$ (доказано в разделе 2.1), это уравнение имеет единственный корень, и

$$\mathrm{CP}(z_t)<\gamma\iff z_t<a_{CP}(t).$$

Численно корень ищут методом Брента (в коде — `brentq`). На плоскости $(t,z)$ правило «стоп по CP» — это кривая $a_{CP}(t)$, ниже которой мы останавливаемся (тот самый красный контур из карты выше).

**Различие типов границ — центральный момент.** CP-граница строится из условного расчёта на каждом взгляде (подставляем наблюдаемое $z_t$) и читается как «вероятность успеха». Но она **не гарантирует заранее**, сколько ошибки II рода будет «съедено» суммарно по всем траекториям. Альтернатива — зафиксировать границу *до* сбора данных с гарантией контроля ошибки II рода. Это и есть β-spending (раздел 5).


In [ ]:
def cp_to_boundary(t: float, gamma: float, theta: float, z_alpha: float = 1.6448536) -> float:
    """z_t, при котором CP(z_t; t, theta) == gamma (численная инверсия монотонной CP)."""
    f = lambda z: conditional_power(z, t, theta, z_alpha) - gamma
    return brentq(f, -8, 8)


In [ ]:
tt_grid = np.linspace(0.05, 0.95, 200)
gamma = 0.15

fig, ax = plt.subplots(figsize=(9, 5))

a_null = [cp_to_boundary(t_, gamma, 0.0, z_alpha) for t_ in tt_grid]
a_mde = [cp_to_boundary(t_, gamma, design.theta_mde, z_alpha) for t_ in tt_grid]

ax.plot(tt_grid, a_null, color=C_CONT, lw=2, label="граница по CP под H0 (θ = 0)")
ax.plot(tt_grid, a_mde, color=C_FUT, lw=2.2, label="граница по CP под MDE")
ax.fill_between(tt_grid, -4, a_mde, color=C_FUT, alpha=0.10)
ax.text(0.6, -1.4, "ниже границы — СТОП (futility)", color=C_FUT, fontsize=11)

z_ex = np.array([1.6, 1.1, 0.55, 0.3])
t_ex = np.array([0.2, 0.5, 0.8, 0.95])
ax.plot(t_ex, z_ex, color="black", lw=1.6, marker="o",
        label="пример наблюдаемой статистики (заходит в зону futility)")

ax.set_xlabel("доля информации t")
ax.set_ylabel("futility-граница $a(t)$ на шкале z")
ax.set_title("Futility-граница из порога CP = 0.15 (данные-зависимая)")
ax.legend(loc="lower left")
plt.show()


## 4. Predictive Power (PPoS): усредняем CP по неопределённости эффекта

Претензия к CP — жёсткая фиксация $\theta$. Подставляя «текущий тренд» $\hat\theta_t$, мы молча притворяемся, что шумная оценка — это истина. Но на раннем взгляде $\hat\theta_t$ очень неточна. **Predictive power (PPoS)** снимает проблему, усредняя CP по всей апостериорной неопределённости эффекта:

$$\mathrm{PPoS} = \int \mathrm{CP}(\theta)\,\pi(\theta \mid D_t)\,d\theta.$$

Интуиция: вместо «каковы шансы при одном фиксированном θ» мы спрашиваем «каковы шансы в среднем по всем правдоподобным θ, взвешенным их апостериорной вероятностью».

<details>
<summary>4.1. Вывод апостериорного распределения (развернуть)</summary>

Априор $\theta\sim\mathcal N(\theta_0,\tau^2)$: $\theta_0$ — исходное представление о типичном эффекте, $\tau$ — насколько мы в нём уверены (малое $\tau$ = сильный prior). Правдоподобие $Z_t\mid\theta\sim\mathcal N(\theta\sqrt t,1)$ эквивалентно наблюдению $y=z_t/\sqrt t$ с моделью $y\mid\theta\sim\mathcal N(\theta,1/t)$. Плотности:

$$\pi(\theta)\propto\exp\!\left(-\frac{(\theta-\theta_0)^2}{2\tau^2}\right),\qquad L(\theta)\propto\exp\!\left(-\frac{t(\theta-y)^2}{2}\right).$$

Апостериор $\propto\pi L$. Логарифмируем и раскрываем квадраты:

$$-2\log = \frac{(\theta-\theta_0)^2}{\tau^2}+t(\theta-y)^2 = \underbrace{\left(\frac1{\tau^2}+t\right)}_{=:A}\theta^2-2\underbrace{\left(\frac{\theta_0}{\tau^2}+ty\right)}_{=:B}\theta+\text{const}.$$

Дополняем до полного квадрата: $A\theta^2-2B\theta=A(\theta-B/A)^2+\text{const}$. Значит апостериор нормален с точностью $A$ и средним $B/A$:

$$\frac1{\sigma_p^2}=\frac1{\tau^2}+t,\qquad \mu_p=\sigma_p^2\left(\frac{\theta_0}{\tau^2}+ty\right).$$

Подставим $ty=t\cdot(z_t/\sqrt t)=\sqrt t\,z_t$:

$$\boxed{\frac1{\sigma_p^2}=\frac1{\tau^2}+t,\qquad \mu_p=\sigma_p^2\left(\frac{\theta_0}{\tau^2}+\sqrt t\,z_t\right)}$$

Ключевая идея: **точности складываются** ($1/\sigma_p^2 = 1/\tau^2 + t$), а $\mu_p$ — взвешенная сумма prior $\theta_0$ и данных $z_t/\sqrt t$ с весами-точностями. Чем больше данных (больше $t$), тем сильнее апостериор «прижимается» к данным.

</details>

<details>
<summary>4.2. Замкнутая форма PPoS (развернуть)</summary>

Числитель CP линеен по $\theta$. Перепишем с $\theta=\mu_p+W$, где $W\sim\mathcal N(0,\sigma_p^2)$:

$$z_{1-\alpha}-\sqrt t\,z_t-\theta(1-t)=z_{1-\alpha}-\sqrt t\,z_t-\mu_p(1-t)-W(1-t)=:A-(1-t)W.$$

Обозначим $\widetilde W=(1-t)W\sim\mathcal N\big(0,(1-t)^2\sigma_p^2\big)$, тогда $\mathrm{CP}(\theta)=1-\Phi\!\left(\frac{A-\widetilde W}{\sqrt{1-t}}\right)$.

**Тождество для усреднения.** Для $W\sim\mathcal N(0,\sigma_W^2)$ и независимого $U\sim\mathcal N(0,1)$:

$$\Phi\!\left(\frac{a-W}{b}\right)=P\!\left(U\le\frac{a-W}{b}\right).$$

Беря матожидание по $W$ (теорема полной вероятности):

$$\mathbb E_W\left[\Phi\!\left(\frac{a-W}{b}\right)\right]=P(bU+W\le a)=\Phi\!\left(\frac{a}{\sqrt{b^2+\sigma_W^2}}\right),$$

так как $bU+W\sim\mathcal N(0,\,b^2+\sigma_W^2)$.

**Применяем.** Здесь $a=A$, $b=\sqrt{1-t}$, $\sigma_W^2=(1-t)^2\sigma_p^2$:

$$\mathrm{PPoS}=1-\Phi\!\left(\frac{A}{\sqrt{(1-t)+(1-t)^2\sigma_p^2}}\right).$$

$$\boxed{\mathrm{PPoS}=1-\Phi\!\left(\frac{z_{1-\alpha}-\sqrt t\,z_t-\mu_p(1-t)}{\sqrt{(1-t)+(1-t)^2\sigma_p^2}}\right)}$$

**Смысл.** Знаменатель больше, чем у CP ($\sqrt{1-t}$), на слагаемое $(1-t)^2\sigma_p^2$, отвечающее за неопределённость эффекта, — поэтому PPoS ниже CP-под-трендом. CP — частный случай PPoS с точечным prior $\sigma_p^2\to0$.

</details>

**Общая рамка.** CP и PPoS — две инкарнации **stochastic curtailment**: останавливаем тест, когда условная вероятность полезного финального исхода при некоем предположении пренебрежимо мала. CP — частотная (фиксированный θ), PPoS — байесовская (усреднение по апостериору). Для общения с бизнесом PPoS удобнее: это непосредственно «вероятность успеха в процентах».


In [ ]:
def ppos(z_t: float, t: float, theta0: float, tau: float, z_alpha: float = 1.6448536) -> float:
    """Predictive probability of success (нормальный prior, замкнутая форма Spiegelhalter–Freedman–Parmar)."""
    sigma2 = 1.0 / (1.0 / tau**2 + t)              # апостериорная дисперсия
    mu = sigma2 * (theta0 / tau**2 + np.sqrt(t) * z_t)   # апостериорное среднее
    return 1.0 - norm.cdf((z_alpha - np.sqrt(t) * z_t - mu * (1 - t))
                          / np.sqrt((1 - t) + (1 - t)**2 * sigma2))


In [ ]:
z_t, tt = 0.4, 0.5
theta0, tau = 0.0, 0.8            # скептический prior: эффект в среднем = 0
sigma2 = 1.0 / (1.0 / tau**2 + tt)
mu = sigma2 * (theta0 / tau**2 + np.sqrt(tt) * z_t)

theta_grid = np.linspace(-0.5, 4, 400)
post = norm.pdf(theta_grid, loc=mu, scale=np.sqrt(sigma2))          # апостериор
cp_theta = conditional_power(z_t, tt, theta_grid, z_alpha)          # CP(θ)
pp = ppos(z_t, tt, theta0, tau, z_alpha)                            # PPoS
cp_trend = conditional_power(z_t, tt, z_t / np.sqrt(tt), z_alpha)   # CP по тренду

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), gridspec_kw={"width_ratios": [1, 1.35]})

axes[0].plot(theta_grid, post, color=C_CONT, lw=2)
axes[0].fill_between(theta_grid, post, color=C_CONT, alpha=0.15)
axes[0].axvline(mu, color="black", ls="--", lw=1.2, label=f"апостериорное среднее {mu:.2f}")
axes[0].axvline(design.theta_mde, color=C_EFF, ls=":", lw=1.2, label=f"MDE-дрейф {design.theta_mde:.2f}")
axes[0].set_title("Апостериор эффекта θ | данные\n(скептический prior N(0, 0.8²))")
axes[0].set_xlabel("θ"); axes[0].set_ylabel("плотность"); axes[0].legend(fontsize=9)

axes[1].plot(theta_grid, cp_theta, color=C_EFF, lw=2.2, label="CP(θ)")
axes[1].fill_between(theta_grid, cp_theta * post, alpha=0.35, color=C_FUT,
                     label="подынтегральное CP(θ)·π(θ)")
axes[1].axhline(pp, color=C_FUT, ls="--", lw=1.6, label=f"PPoS = {pp:.3f}")
axes[1].axhline(cp_trend, color=C_CONT, ls="--", lw=1.6, label=f"CP(тренд) = {cp_trend:.3f}")
axes[1].set_title("PPoS = среднее CP(θ) по апостериору")
axes[1].set_xlabel("θ"); axes[1].set_ylabel("вероятность"); axes[1].legend(fontsize=9)

fig.suptitle("PPoS «притянут» к массе апостериора у нуля → ниже, чем CP(тренд)", fontsize=13.5)
fig.tight_layout()
plt.show()


In [ ]:
def plot_ppos_interactive():
    def _plot(t, z_t, theta0, tau):
        pp = ppos(z_t, t, theta0, tau, z_alpha)
        cpt = conditional_power(z_t, t, z_t / np.sqrt(t), z_alpha)
        fig, ax = plt.subplots(figsize=(8, 2.6))
        ax.barh([1, 0], [pp, cpt], color=[C_FUT, C_CONT])
        ax.set_yticks([0, 1]); ax.set_yticklabels(["CP(тренд)", "PPoS"])
        ax.set_xlim(0, 1); ax.axvline(0.15, ls="--", color="black")
        ax.set_title(f"PPoS = {pp:.3f}   vs   CP(тренд) = {cpt:.3f}")
        plt.show()
    interact(_plot,
             t=FloatSlider(value=0.5, min=0.05, max=0.95, step=0.05, description="t"),
             z_t=FloatSlider(value=0.4, min=-3, max=3, step=0.05, description="z_t"),
             theta0=FloatSlider(value=0.0, min=-1, max=3, step=0.1, description="prior θ0"),
             tau=FloatSlider(value=0.8, min=0.05, max=3, step=0.05, description="prior τ"))
plot_ppos_interactive()


## 5. β-spending: заранее фиксированная futility-граница

CP давал границу, которая «плавает» вместе с данными. Минус — мы не можем заранее гарантировать, сколько ошибки II рода она «съест». Альтернатива — построить futility-границу **до начала теста**, как мы строим efficacy-границу. Инструмент — **β-spending**, точное зеркало α-spending.

**Логика симметрии.** α-spending распределяет бюджет ошибки I рода по взглядам под $H_0$; β-spending — то же для ошибки II рода: «распределить β так, чтобы суммарная вероятность *зря остановить тест при реальном эффекте* была не больше β». Заметьте тонкость: «зря остановить» возможно только когда эффект реально есть, поэтому futility-границу считают **под альтернативой** $H_1$ ($\theta=\theta_{MDE}$), а efficacy — под $H_0$.

Спендинг-функции — неубывающие функции доли информации $t$: $\alpha(0)=\beta(0)=0$, $\alpha(1)=\alpha$, $\beta(1)=\beta$. Степенное семейство $\beta(t)=\beta t^{\rho}$: $\rho=3$ — OBF-подобная (консервативно рано), $\rho=1$ — Pocock-подобная (агрессивно рано).

<details>
<summary>5.1. Рекурсивный расчёт границ — полный алгоритм (развернуть)</summary>

Работаем на шкале $B$. Обозначим приращения бюджетов:

$$\Delta\alpha_k=\alpha(t_k)-\alpha(t_{k-1}),\qquad \Delta\beta_k=\beta(t_k)-\beta(t_{k-1}).$$

**Плотность продолжения.** Пусть $p_k(x)$ — (суб)плотность $B(t_k)$ среди траекторий, «доживших» до взгляда $k-1$, не пересекнув ни одну границу. Её интеграл — вероятность ещё не остановиться к этому моменту.

**Инициализация** ($k=1$, приращение от 0 до $t_1$, дисперсия $t_1$):

- для efficacy, под $H_0$: $\;p_1^{eff}(x)=\varphi(x;\,0,\,t_1)$;
- для futility, под $H_1$: $\;p_1^{fut}(x)=\varphi(x;\,\theta t_1,\,t_1)$.

**Шаг отсечения (взгляд $k$).** Efficacy-граница $b_k$ отсекает правый хвост с массой $\Delta\alpha_k$:

$$\int_{b_k}^{\infty}p_k^{eff}(x)\,dx=\Delta\alpha_k.$$

Futility-граница $a_k$ отсекает левую часть с массой $\Delta\beta_k$ (уже зная $b_k$):

$$\int_{-\infty}^{a_k}p_k^{fut}(x)\,dx=\Delta\beta_k.$$

**Шаг свёртки (между взглядами).** Оставляем только траектории в области продолжения и добавляем независимое приращение $B(t_{k+1})-B(t_k)\sim\mathcal N(\theta\Delta_{k+1},\,\Delta_{k+1})$, где $\Delta_{k+1}=t_{k+1}-t_k$:

$$p_{k+1}^{fut}(y)=\int_{a_k}^{b_k}p_k^{fut}(x)\,\varphi\big(y;\,x+\theta\Delta_{k+1},\,\Delta_{k+1}\big)\,dx,$$

и аналогично для efficacy (при non-binding нижней границы нет, то есть $a_k=-\infty$).

**Почему это работает.** Урезание оставляет только «ещё продолжающиеся» траектории, а свёртка прибавляет следующий независимый шаг. Независимость приращений (раздел 1.4) гарантирует, что распределение следующего шага зависит только от текущей позиции, а не от всей истории, — поэтому рекурсия точна (с точностью до дискретизации сетки $x$). Каждый шаг тратит ровно приращение бюджета $\Delta\alpha_k$ или $\Delta\beta_k$; суммируясь по $k$, они дают в точности $\alpha$ и $\beta$.

**Возврат на шкалу $Z$:** $c_k=b_k/\sqrt{t_k}$, $\;a_k^{(\text{Z})}=a_k/\sqrt{t_k}$.

</details>


In [ ]:
def alpha_spending(t, alpha: float = 0.05, family: str = "obf") -> np.ndarray:
    """Кумулятивная α-spending функция (Lan–DeMets)."""
    t = np.asarray(t, dtype=float)
    if family == "obf":
        return 2 - 2 * norm.cdf(norm.ppf(1 - alpha / 2) / np.sqrt(t))
    if family == "pocock":
        return alpha * np.log(1 + (np.e - 1) * t)
    raise ValueError(family)


def beta_spending(t, beta: float = 0.20, family: str = "obf") -> np.ndarray:
    """Кумулятивная β-spending функция (степенное семейство Kim–DeMets)."""
    t = np.asarray(t, dtype=float)
    if family == "obf":
        return beta * t ** 3          # ρ=3 — OBF-подобная
    if family == "pocock":
        return beta * t               # ρ=1 — Pocock-подобная
    raise ValueError(family)


In [ ]:
def _convolve(p: np.ndarray, grid: np.ndarray, loc: float, scale: float) -> np.ndarray:
    """Свёртка плотности с нормальным приращением N(loc, scale^2).

    p_next(x_j) = Σ_i p(x_i) * φ(x_j; x_i + loc, scale) * dx.
    """
    dx = grid[1] - grid[0]
    K = norm.pdf(grid[:, None], loc=grid[None, :] + loc, scale=scale)
    return (K @ p) * dx


def _quantile(p: np.ndarray, grid: np.ndarray, target: float, tail: bool) -> float:
    """Точка, отсекающая массу target: правый хвост (tail=True) или левую часть (tail=False).

    Для субплотности с общей массой cdf[-1] правый хвост = cdf[-1] - cdf(x).
    """
    dx = grid[1] - grid[0]
    cdf = np.cumsum(p) * dx
    vals = (cdf[-1] - cdf) if tail else cdf
    idx = int(np.argmin(np.abs(vals - target)))
    return grid[idx]


def group_sequential_boundaries(t, alpha=0.05, beta=0.20, theta=None,
                                fam_alpha="obf", fam_beta="obf",
                                grid_dx=0.01, n_sigma=6.0):
    """Односторонние efficacy (верхняя c) и futility (нижняя a) границы на шкале Z.

    Efficacy-граница считается под H0 (θ=0) без нижней границы (non-binding);
    futility-граница — под H1 (θ=θ_mde) с обеими границами в области продолжения.
    """
    t = np.asarray(t, dtype=float)
    K = len(t)
    if theta is None:
        theta = norm.ppf(1 - alpha) + norm.ppf(1 - beta)
    dt = np.diff(np.concatenate(([0.0], t)))
    grid = np.arange(-n_sigma, n_sigma + grid_dx, grid_dx)

    d_alpha = np.diff(np.concatenate(([0.0], alpha_spending(t, alpha, fam_alpha))))
    d_beta = np.diff(np.concatenate(([0.0], beta_spending(t, beta, fam_beta))))

    c = np.empty(K); a = np.empty(K)
    p_eff = norm.pdf(grid, loc=0.0, scale=math.sqrt(dt[0]))            # плотность B_1 под H0
    p_fut = norm.pdf(grid, loc=theta * dt[0], scale=math.sqrt(dt[0]))  # плотность B_1 под H1
    b_prev, a_prev = np.inf, -np.inf
    for k in range(K):
        b_k = _quantile(p_eff, grid, d_alpha[k], tail=True)
        c[k] = b_k / math.sqrt(t[k]); b_prev = b_k
        a_k = _quantile(p_fut, grid, d_beta[k], tail=False)
        a[k] = a_k / math.sqrt(t[k]); a_prev = a_k
        if k < K - 1:
            d = dt[k + 1]
            p_eff = _convolve(p_eff * (grid <= b_k), grid, 0.0, math.sqrt(d))
            p_fut = _convolve(p_fut * ((grid > a_k) & (grid <= b_k)), grid, theta * d, math.sqrt(d))
    return c, a


c_bound, a_bound = group_sequential_boundaries(design.t, design.alpha, design.beta, design.theta_mde)
boundaries = pd.DataFrame({
    "взгляд k": np.arange(1, design.K + 1),
    "t_k": design.t,
    "c_k (efficacy)": np.round(c_bound, 3),
    "a_k (futility)": np.round(a_bound, 3),
})
boundaries


In [ ]:
tt = np.asarray(design.t)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))

a_sp = alpha_spending(tt, design.alpha, "obf")
axes[0].plot(tt, a_sp, marker="o", color=C_EFF, lw=2.2)
axes[0].fill_between(tt, a_sp, color=C_EFF, alpha=0.18)
axes[0].axhline(design.alpha, ls="--", color="black", lw=1)
axes[0].text(0.02, design.alpha + 0.006, "бюджет α = 0.05", fontsize=10)
axes[0].set_title("α-spending (O'Brien–Fleming)\nна efficacy-границу")
axes[0].set_xlabel("t"); axes[0].set_ylabel("накопленная ошибка I рода")

b_sp = beta_spending(tt, design.beta, "obf")
axes[1].plot(tt, b_sp, marker="o", color=C_FUT, lw=2.2)
axes[1].fill_between(tt, b_sp, color=C_FUT, alpha=0.18)
axes[1].axhline(design.beta, ls="--", color="black", lw=1)
axes[1].text(0.02, design.beta + 0.006, "бюджет β = 0.20", fontsize=10)
axes[1].set_title("β-spending (ρ = 3)\nна futility-границу")
axes[1].set_xlabel("t"); axes[1].set_ylabel("накопленная ошибка II рода")

fig.suptitle("Две стороны одной монеты: как «тратится» ошибка I и II рода по ходу теста", fontsize=13.5)
fig.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
tt = np.asarray(design.t)

ax.plot(tt, c_bound, color=C_EFF, lw=2.2, marker="o", label="efficacy-граница $c_k$")
ax.plot(tt, a_bound, color=C_FUT, lw=2.2, marker="o", label="futility-граница $a_k$")

ax.fill_between(tt, c_bound, 4.2, color=C_EFF, alpha=0.12)
ax.fill_between(tt, -4.2, a_bound, color=C_FUT, alpha=0.12)
ax.fill_between(tt, a_bound, c_bound, color=C_CONT, alpha=0.06)

ax.text(0.66, 3.4, "СТОП — эффект\n(Z > c_k)", color=C_EFF, fontsize=12, ha="center", weight="bold")
ax.text(0.66, -2.2, "СТОП — futility\n(Z < a_k)", color=C_FUT, fontsize=12, ha="center", weight="bold")
ax.text(0.66, 0.8, "продолжаем", color="black", fontsize=12, ha="center")

ax.axhline(0, color="black", lw=0.8)
ax.set_xlabel("доля информации t")
ax.set_ylabel("Z(t)")
ax.set_ylim(-4.2, 4.2)
ax.set_title("Двухграничный групповой-последовательный дизайн")
ax.legend(loc="upper right")
plt.show()


### 5.2. Расширение на двустороннюю проверку

До сих пор был односторонний тест, где futility означает «эффект не догонит MDE в выгодную сторону». Для **двустороннего** теста (интересно отклонение в любую сторону) всё зеркально:

- **efficacy**: останавливаемся при $|Z_t| > c_k$ — граница симметрична, бюджет α делится на два хвоста по $\alpha/2$;
- **futility**: останавливаемся, когда эффект «зажат» в симметричном коридоре $|Z_t| < a_k$ — величина эффекта слишком мала, чтобы дотянуться до значимости в любую из сторон.

Геометрически это та же картинка, отражённая ниже нуля: futility — «воронка» вокруг нуля, efficacy — два симметричных хвоста.


In [ ]:
z_alpha2 = norm.ppf(1 - design.alpha / 2)
theta2 = z_alpha2 + norm.ppf(1 - design.beta)   # дрейф для двустороннего MDE
c2, a2 = group_sequential_boundaries(design.t, design.alpha, design.beta, theta2)

fig, ax = plt.subplots(figsize=(9, 5.5))
tt = np.asarray(design.t)

ax.plot(tt, c2, color=C_EFF, lw=2.2, marker="o", label="± efficacy (двусторонняя)")
ax.plot(tt, -c2, color=C_EFF, lw=2.2, marker="o")
ax.plot(tt, a2, color=C_FUT, lw=2.2, marker="o", label="± futility (двусторонняя)")
ax.plot(tt, -a2, color=C_FUT, lw=2.2, marker="o")

ax.fill_between(tt, a2, -a2, color=C_FUT, alpha=0.12)
ax.fill_between(tt, c2, 4.2, color=C_EFF, alpha=0.12)
ax.fill_between(tt, -4.2, -c2, color=C_EFF, alpha=0.12)

ax.text(0.66, 3.4, "эффект (|Z| > c_k)", color=C_EFF, fontsize=12, ha="center", weight="bold")
ax.text(0.66, 0.0, "futility (|Z| < a_k)", color=C_FUT, fontsize=12, ha="center", weight="bold")
ax.axhline(0, color="black", lw=0.8)
ax.set_xlabel("доля информации t"); ax.set_ylabel("Z(t)")
ax.set_ylim(-4.2, 4.2)
ax.set_title("Двусторонний дизайн: futility = коридор |Z| < a_k вокруг нуля")
ax.legend(loc="upper right")
plt.show()


## 6. Binding vs non-binding futility

Futility-граница влияет на то, какие траектории доживают до конца. Отсюда тонкий вопрос: **как учитывать её при расчёте ошибки I рода α?**

- **Binding (обязывающая)**: futility-стоп обязателен, и α считается с учётом удаления траекторий ниже $a_k$. Если потом проигнорировать стоп, отброшенные траектории вернутся, и α вырастет.
- **Non-binding (необязывающая)**: α считается так, *будто futility-границы нет*. Тогда стоп можно отменить в любой момент без риска для α.

<details>
<summary>Почему binding α не больше non-binding (развернуть)</summary>

В non-binding расчёте на каждом взгляде область продолжения — «ещё не пересекли efficacy». В binding она дополнительно ограничена снизу: $a_j<Z\le c_j$ для всех $j<k$. Продолжающееся множество в binding — **подмножество** non-binding, поэтому масса, достигающая efficacy, не больше. Отсюда же: игнорируя binding-стоп, мы расширяем множество обратно до non-binding — и α возвращается к более высокому значению.

</details>

**Почему non-binding — стандарт.** Futility — рекомендация, а не приговор: аналитик может продолжить тест после шумного первого взгляда. Non-binding формально разрешает эту гибкость, не ломая контроль α. Поэтому в продуктовых A/B почти всегда используют non-binding.


In [ ]:
def overall_alpha_upper(c: np.ndarray, t: np.ndarray, lower=None, n_sim: int = 500000):
    """Monte-Carlo: P(хотя бы один взгляд Z_k > c_k) под H0 (θ=0).

    lower=None  → futility-границы нет (non-binding, α считаем как без неё);
    lower=a_k   → binding: остановленные по futility исключаются.
    """
    dt = np.diff(np.concatenate(([0.0], t)))
    B = np.cumsum(rng.normal(0, np.sqrt(dt), size=(n_sim, len(t))), axis=1)
    Z = B / np.sqrt(t)
    active = np.ones(n_sim, dtype=bool)
    crossed = np.zeros(n_sim, dtype=bool)
    for k in range(len(t)):
        if lower is not None:
            active &= ~(Z[:, k] < lower[k]) & ~crossed
        crossed |= active & (Z[:, k] > c[k])
        active &= ~crossed
    return crossed.mean()


t = np.asarray(design.t)
alpha_none = overall_alpha_upper(c_bound, t, lower=None)
alpha_bind = overall_alpha_upper(c_bound, t, lower=a_bound)

res = pd.DataFrame({
    "Сценарий": ["без futility-границы (non-binding)", "binding"],
    "Суммарный α (efficacy)": np.round([alpha_none, alpha_bind], 4),
})
res["Отличие от номинального α = 0.05"] = np.round(res["Суммарный α (efficacy)"] - design.alpha, 4)
res


## 7. Влияние futility на ошибки I/II рода + Monte-Carlo

Теперь самый важный факт лекции, ради которого всё и затевалось.

**Почему futility НЕ раздувает α.** Ошибка I рода — траектория, которая под $H_0$ забралась выше efficacy-границы. Futility отбрасывает траектории **ниже** себя (слабо выглядящие); они и так редко достигали бы efficacy, а futility-стоп лишь *уменьшает* их число. Поэтому вероятность отвергнуть при $H_0$ может только уменьшиться. Заметьте: аргумент использует только вложенность и монотонность, а **не** нормальность — это важное замечание, к нему вернёмся в разделе 9.

**Почему futility снижает мощность.** Под $H_1$ (эффект реально есть) часть траекторий на раннем взгляде *случайно* выглядит слабо — futility зря их останавливает. Это «неудачные заходы» — плата за экономию трафика.

Проверим оба утверждения симуляцией и измерим операционные характеристики: мощность, вероятность futility-стопа под H0/H1 и средний размер выборки (ASN — average sample number).


In [ ]:
def simulate_design(theta, c, a, t, n_sim=50000):
    """MC группо-последовательного дизайна. Исходы: efficacy / futility / complete.

    Симулируем приращения B: B(t_k) = Σ_{j≤k} ε_j, ε_j ~ N(theta*Δ_j, Δ_j).
    """
    dt = np.diff(np.concatenate(([0.0], t)))
    B = np.cumsum(rng.normal(theta * dt, np.sqrt(dt), size=(n_sim, len(t))), axis=1)
    Z = B / np.sqrt(t)
    outcome = np.full(n_sim, "complete")
    stop_frac = np.ones(n_sim)
    for k in range(len(t)):
        active = outcome == "complete"
        if not active.any():
            break
        eff = active & (Z[:, k] > c[k])
        fut = active & (Z[:, k] < a[k])
        outcome[eff] = "efficacy"; outcome[fut] = "futility"
        stop_frac[eff] = t[k]; stop_frac[fut] = t[k]
    return outcome, stop_frac


def oc(theta, c, a, t, n_sim=50000):
    """Операционные характеристики: (power, P(futility), P(complete), ASN-доля)."""
    out, stop = simulate_design(theta, c, a, t, n_sim)
    p_eff = (out == "efficacy").mean()
    p_fut = (out == "futility").mean()
    p_comp = (out == "complete").mean()
    return p_eff, p_fut, p_comp, stop.mean()


t = np.asarray(design.t)
p0 = oc(0.0, c_bound, a_bound, t)
p1 = oc(design.theta_mde, c_bound, a_bound, t)
print(f"Под H0 (θ=0):     FPR={p0[0]:.4f}  futility-stop={p0[1]:.3f}  complete={p0[2]:.3f}  E[N]/Nmax={p0[3]:.3f}")
print(f"Под H1 (θ=MDE):   power={p1[0]:.4f}  futility-stop={p1[1]:.3f}  complete={p1[2]:.3f}  E[N]/Nmax={p1[3]:.3f}")

# Цена futility: мощность с futility-границей против мощности без неё
a_none = np.full(len(t), -np.inf)
p1_nofut = oc(design.theta_mde, c_bound, a_none, t)
print(f"\nМощность без futility (только efficacy): {p1_nofut[0]:.4f}")
print(f"Мощность с futility:                      {p1[0]:.4f}")
print(f"Потеря мощности от futility:              {p1_nofut[0] - p1[0]:.4f}")


In [ ]:
# Операционные характеристики по спектру эффектов + доверительные полосы MC.
thetas = np.linspace(-0.2, 3.0, 17)
N = 30000
rows = [oc(th, c_bound, a_bound, t, N) for th in thetas]
oc_df = pd.DataFrame(rows, columns=["power", "futility_stop", "complete", "asn"])
oc_df["theta"] = thetas
se = np.sqrt(oc_df[["power", "futility_stop", "complete"]].mul(1 - oc_df[["power", "futility_stop", "complete"]]) / N)

fig, ax = plt.subplots(figsize=(9, 5.2))
ax.plot(thetas, oc_df["power"], color=C_EFF, lw=2.2, label="power (дошли до efficacy)")
ax.plot(thetas, oc_df["futility_stop"], color=C_FUT, lw=2.2, label="P(stop по futility)")
ax.plot(thetas, oc_df["complete"], color=C_CONT, lw=2.2, label="P(дошли до конца)")
for col, color in [("power", C_EFF), ("futility_stop", C_FUT), ("complete", C_CONT)]:
    ax.fill_between(thetas, oc_df[col] - 2 * se[col], oc_df[col] + 2 * se[col],
                    color=color, alpha=0.12)

ax.axvline(design.theta_mde, color="black", ls="--", lw=1.3)
ax.text(design.theta_mde + 0.05, 0.9, "θ_MDE", fontsize=11)
ax.set_xlabel("истинный дрейф θ")
ax.set_ylabel("вероятность исхода")
ax.set_title("Операционные характеристики: сумма трёх кривых = 1")
ax.legend(loc="center right")
plt.show()


In [ ]:
# Ансамбль траекторий: наглядно видно, как futility-граница "отфильтровывает"
# бесперспективные заходы под H0 и изредка зря останавливает тест под H1.
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
cases = [(0.0, "H0: почти всё уходит в futility — это выгодная экономия"),
         (design.theta_mde, "H1: большинство доходит до efficacy, часть «застревает»")]

for ax, (theta, title) in zip(axes, cases):
    dt = np.diff(np.concatenate(([0.0], t)))
    B = np.cumsum(rng.normal(theta * dt, np.sqrt(dt), size=(250, len(t))), axis=1)
    Z = B / np.sqrt(t)
    for i in range(250):
        col = C_CONT
        for k in range(len(t)):
            if Z[i, k] > c_bound[k]:
                col = C_EFF; break
            if Z[i, k] < a_bound[k]:
                col = C_FUT; break
        ax.plot(t, Z[i], color=col, alpha=0.3, lw=0.8)
    ax.plot(t, c_bound, color=C_EFF, lw=2.4)
    ax.plot(t, a_bound, color=C_FUT, lw=2.4)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("t")

axes[0].set_ylabel("Z(t)")
from matplotlib.lines import Line2D
legend = [Line2D([0], [0], color=C_EFF, lw=2.4, label="efficacy"),
          Line2D([0], [0], color=C_FUT, lw=2.4, label="futility"),
          Line2D([0], [0], color=C_CONT, lw=2, alpha=0.5, label="дошли до конца")]
fig.legend(handles=legend, loc="lower center", ncol=3, fontsize=11)
fig.suptitle("Ансамбль траекторий, раскрашенный по исходу: видно случайное время остановки", fontsize=13.5)
fig.tight_layout(rect=[0, 0.06, 1, 0.96])
plt.show()


## 8. Практические рекомендации (следствия из математики)

Соберём всё в чек-лист дизайнера. Каждый пункт — не догма, а следствие того, что мы разобрали выше.

1. **Взгляды — не раньше 30–50% данных.** При малом $t$ карта CP почти плоская (будущее решает судьбу), поэтому ранний futility-стоп почти ничего «не знает» и зря теряет мощность. Поздние взгляды почти бесплатные.
2. **Порог CP/PPoS — 10–20%.** Выше — заметная потеря мощности, ниже 5% — экономии почти нет. Это Парето-компромисс «мощность ↔ трафик».
3. **Выбор θ для CP** — как в разделе 2.
4. **Non-binding — дефолт:** futility-стоп можно отменить, не испортив контроль α.
5. **Полный дизайн = α-spending + β-spending:** α-бюджет строит efficacy-границу, β-бюджет — futility-границу; вместе — контролируемый по обеим ошибкам групповой-последовательный дизайн.

Интерактив ниже собирает всё вместе: крутите число взглядов и формы спендинга — и смотрите, как меняются границы и операционные характеристики.


In [ ]:
def design_full(alpha=0.05, beta=0.20, K=5, fam_alpha="obf", fam_beta="obf"):
    t = np.linspace(0.3, 1.0, K)
    theta = norm.ppf(1 - alpha) + norm.ppf(1 - beta)
    c, a = group_sequential_boundaries(t, alpha, beta, theta, fam_alpha, fam_beta)
    p0 = oc(0.0, c, a, t, 20000)
    p1 = oc(theta, c, a, t, 20000)
    return t, c, a, p0, p1


def plot_design_dashboard(K, fam_alpha, fam_beta):
    t, c, a, p0, p1 = design_full(0.05, 0.20, K, fam_alpha, fam_beta)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
    axes[0].plot(t, c, color=C_EFF, lw=2.2, marker="o", label="efficacy")
    axes[0].plot(t, a, color=C_FUT, lw=2.2, marker="o", label="futility")
    axes[0].fill_between(t, a, c, color=C_CONT, alpha=0.06)
    axes[0].set_title("Границы"); axes[0].set_xlabel("t"); axes[0].legend()
    axes[1].bar(["FPR (H0)", "power (H1)", "futility (H1)"],
                [p0[0], p1[0], p1[1]], color=[C_CONT, C_EFF, C_FUT])
    axes[1].set_ylim(0, 1)
    axes[1].set_title(f"OC: E[N]/Nmax = {p1[3]:.2f} (под H1)")
    fig.tight_layout(); plt.show()

interact(plot_design_dashboard,
         K=IntSlider(value=3, min=2, max=8, step=1, description="взглядов K"),
         fam_alpha=Dropdown(options=["obf", "pocock"], value="obf", description="α-spending"),
         fam_beta=Dropdown(options=["obf", "pocock"], value="obf", description="β-spending"))


## 9. Ненормальность метрики (краткая сводка)

Вся математика опирается на нормальность **статистики**, а не самой метрики — это следствие центральной предельной теоремы (ЦПТ). Полезно разделить два ингредиента, у которых разная судьба:

- **Независимость приращений** — свойство *дизайна*, оно точное и от распределения метрики не зависит. Ломается только кластерами или carryover-эффектами (и чинится дизайном).
- **Нормальность $Z_t$** — *асимптотика*. При «приятной» метрике и достаточном $n$ границы валидны с точностью $O(1/\sqrt n)$. При тяжёлых хвостах и малом $n$ — смещение; лечится студентизацией, information-based monitoring или симуляционным дизайном по реалистичному распределению.

Качественные факты («futility не раздувает α», логика β-spending, non-binding) от нормальности **не зависят** — это важно, потому что именно они и есть главный результат лекции.


## 10. Итоги

1. **Futility не раздувает ошибку I рода** — в отличие от peeking по эффективности; она лишь снижает мощность.
2. **CP** — шанс дойти до значимости при фиксированном θ; **PPoS** усредняет по неопределённости θ; CP — частный случай PPoS.
3. **Stochastic curtailment** — общая рамка CP и PPoS.
4. **β-spending — зеркало α-spending**: априорная futility-граница с контролем ошибки II рода.
5. **Non-binding** — безопасный дефолт; тайминг, порог и prior определяют компромисс «мощность ↔ экономия трафика».

| | CP | PPoS | β-spending |
|---|---|---|---|
| Философия | частотная | байесовская | частотная (дизайн) |
| Эффект | фиксированный θ | prior на θ | θ_MDE |
| Граница | данные-зависимая | данные-зависимая | фиксирована заранее |
| Контролирует | правило «стоп» | правило «стоп» | ошибку II рода |
| Когда | быстро оценить шансы | учесть неопределённость | спроектировать тест |
